# Previsão de Risco de AVC — Tech Challenge Fase 1
**PosTech FIAP — IA para Devs**

**Dataset:** Stroke Prediction Dataset (Kaggle/UCI)  
**Objetivo:** Classificar pacientes em risco de AVC utilizando técnicas de Machine Learning supervisionado.  
**Modelos:** Regressão Logística (baseline), Random Forest e Gradient Boosting  
**Métrica principal:** Recall (minimizar falsos negativos num contexto clínico)

---

## 1. Imports e Configurações

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score,
    roc_curve, f1_score, recall_score, precision_score, accuracy_score,
    ConfusionMatrixDisplay
)
from sklearn.utils import resample

# Estilo dos gráficos
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 100
plt.rcParams['figure.figsize'] = (10, 6)

RANDOM_STATE = 42
print('Bibliotecas carregadas com sucesso!')

---
## 2. Carregamento e Exploração dos Dados (EDA)

In [ ]:
df = pd.read_csv('../data/healthcare-dataset-stroke-data.csv')
print(f'Shape: {df.shape}')
df.head(10)

In [ ]:
print('=== Tipos e Nulos ===')
info = pd.DataFrame({
    'Tipo': df.dtypes,
    'Nulos': df.isnull().sum(),
    '% Nulos': (df.isnull().sum() / len(df) * 100).round(2),
    'Únicos': df.nunique()
})
print(info)

In [ ]:
df.describe()

### 2.1 Distribuição da variável alvo (stroke)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

counts = df['stroke'].value_counts()
colors = ['#2ecc71', '#e74c3c']
axes[0].bar(['Sem AVC (0)', 'Com AVC (1)'], counts.values, color=colors, edgecolor='white', linewidth=1.5)
axes[0].set_title('Distribuição da Variável Alvo', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Quantidade de Pacientes')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 30, str(v), ha='center', fontweight='bold')

axes[1].pie(counts.values, labels=['Sem AVC', 'Com AVC'], colors=colors,
            autopct='%1.1f%%', startangle=90, textprops={'fontsize': 12})
axes[1].set_title('Proporção da Variável Alvo', fontsize=14, fontweight='bold')

plt.suptitle(f'Desbalanceamento: {counts[1]/len(df)*100:.1f}% dos pacientes tiveram AVC', 
             fontsize=12, style='italic', y=0)
plt.tight_layout()
plt.savefig('../outputs/01_distribuicao_target.png', bbox_inches='tight')
plt.show()
print(f'Classe 0 (Sem AVC): {counts[0]} ({counts[0]/len(df)*100:.1f}%)')
print(f'Classe 1 (Com AVC): {counts[1]} ({counts[1]/len(df)*100:.1f}%)')
print('Dataset severamente desbalanceado — será necessário tratamento!')

### 2.2 Distribuição das variáveis numéricas

In [ ]:
num_cols = ['age', 'avg_glucose_level', 'bmi']
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

for i, col in enumerate(num_cols):
    # Histograma geral
    axes[0, i].hist(df[col].dropna(), bins=40, color='steelblue', edgecolor='white', alpha=0.8)
    axes[0, i].set_title(f'Distribuição: {col}', fontweight='bold')
    axes[0, i].axvline(df[col].mean(), color='red', linestyle='--', label=f'Média: {df[col].mean():.1f}')
    axes[0, i].axvline(df[col].median(), color='orange', linestyle='--', label=f'Mediana: {df[col].median():.1f}')
    axes[0, i].legend(fontsize=9)

    # Por classe
    for stroke_val, color, label in [(0, '#2ecc71', 'Sem AVC'), (1, '#e74c3c', 'Com AVC')]:
        subset = df[df['stroke'] == stroke_val][col].dropna()
        axes[1, i].hist(subset, bins=30, alpha=0.6, color=color, label=label, edgecolor='white')
    axes[1, i].set_title(f'{col} por Classe', fontweight='bold')
    axes[1, i].legend()

plt.suptitle('Distribuição das Variáveis Numéricas', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../outputs/02_distribuicao_numericas.png', bbox_inches='tight')
plt.show()

### 2.3 Distribuição das variáveis categóricas

In [ ]:
cat_cols = ['gender', 'hypertension', 'heart_disease', 'ever_married', 'work_type', 'Residence_type', 'smoking_status']

fig, axes = plt.subplots(3, 3, figsize=(16, 14))
axes = axes.flatten()

for i, col in enumerate(cat_cols):
    stroke_rate = df.groupby(col)['stroke'].mean().sort_values(ascending=False)
    bars = axes[i].bar(stroke_rate.index.astype(str), stroke_rate.values * 100,
                       color=plt.cm.RdYlGn_r(stroke_rate.values / stroke_rate.values.max()),
                       edgecolor='white', linewidth=1.2)
    axes[i].set_title(f'Taxa de AVC por {col}', fontweight='bold', fontsize=11)
    axes[i].set_ylabel('Taxa de AVC (%)')
    axes[i].tick_params(axis='x', rotation=15)
    for bar, val in zip(bars, stroke_rate.values):
        axes[i].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
                     f'{val*100:.1f}%', ha='center', va='bottom', fontsize=9)

for j in range(len(cat_cols), len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Taxa de AVC por Variável Categórica', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../outputs/03_categoricas_vs_stroke.png', bbox_inches='tight')
plt.show()

---
## 3. Pré-processamento de Dados

### 3.1 Limpeza dos dados

In [ ]:
df_clean = df.copy()

# Remover coluna id (não é feature)
df_clean.drop(columns=['id'], inplace=True)
print('Coluna id removida')

# Remover registro 'Other' em gender (apenas 1 caso — outlier)
df_clean = df_clean[df_clean['gender'] != 'Other']
print(f'Registro "Other" em gender removido. Shape: {df_clean.shape}')

# Imputar BMI com mediana (distribuição levemente assimétrica)
bmi_median = df_clean['bmi'].median()
df_clean['bmi'].fillna(bmi_median, inplace=True)
print(f'BMI: {df["bmi"].isnull().sum()} valores nulos imputados com mediana ({bmi_median:.1f})')

# Tratar 'Unknown' em smoking_status como categoria separada (manter — pode ter valor preditivo)
print(f'smoking_status "Unknown" mantido como categoria (1544 registros — valor preditivo relevante)')

print(f'\nShape final: {df_clean.shape}')
print(f'Nulos restantes: {df_clean.isnull().sum().sum()}')

### 3.2 Análise de Correlação

In [ ]:
# Encoding temporário para correlação
df_corr = df_clean.copy()
le = LabelEncoder()
for col in df_corr.select_dtypes(include='object').columns:
    df_corr[col] = le.fit_transform(df_corr[col])

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Heatmap completo
corr_matrix = df_corr.corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, ax=axes[0], linewidths=0.5, annot_kws={'size': 9})
axes[0].set_title('Matriz de Correlação', fontweight='bold', fontsize=13)

# Correlação com target
corr_target = df_corr.corr()['stroke'].drop('stroke').sort_values(key=abs, ascending=True)
colors_bar = ['#e74c3c' if v > 0 else '#3498db' for v in corr_target.values]
axes[1].barh(corr_target.index, corr_target.values, color=colors_bar, edgecolor='white')
axes[1].set_title('Correlação com Variável Alvo (stroke)', fontweight='bold', fontsize=13)
axes[1].axvline(0, color='black', linewidth=0.8)
axes[1].set_xlabel('Correlação de Pearson')

plt.tight_layout()
plt.savefig('../outputs/04_correlacao.png', bbox_inches='tight')
plt.show()
print('\nTop correlações com stroke:')
print(corr_target.sort_values(key=abs, ascending=False).head(8))

### 3.3 Pipeline de pré-processamento e divisão treino/teste

In [ ]:
# Encoding de variáveis categóricas
df_model = df_clean.copy()

# Label Encoding para binárias
binary_cols = ['gender', 'ever_married', 'Residence_type']
le = LabelEncoder()
for col in binary_cols:
    df_model[col] = le.fit_transform(df_model[col])

# One-Hot Encoding para multi-classe
df_model = pd.get_dummies(df_model, columns=['work_type', 'smoking_status'], drop_first=False)

print('Features após encoding:', df_model.shape[1] - 1)
print('Colunas:', list(df_model.columns))

In [ ]:
# Separação features e target
X = df_model.drop(columns=['stroke'])
y = df_model['stroke']

# Divisão estratificada: 70% treino, 15% validação, 15% teste
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.15, stratify=y, random_state=RANDOM_STATE
)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.176, stratify=y_temp, random_state=RANDOM_STATE
)

print(f'Treino:    {X_train.shape[0]} amostras ({X_train.shape[0]/len(X)*100:.1f}%)')
print(f'Validação: {X_val.shape[0]} amostras ({X_val.shape[0]/len(X)*100:.1f}%)')
print(f'Teste:     {X_test.shape[0]} amostras ({X_test.shape[0]/len(X)*100:.1f}%)')
print(f'\nProporção AVC no treino:    {y_train.mean()*100:.2f}%')
print(f'Proporção AVC na validação: {y_val.mean()*100:.2f}%')
print(f'Proporção AVC no teste:     {y_test.mean()*100:.2f}%')

In [ ]:
# Oversampling da classe minoritária no TREINO (sem contaminar val/teste)
X_train_df = X_train.copy()
X_train_df['stroke'] = y_train.values

majority = X_train_df[X_train_df['stroke'] == 0]
minority = X_train_df[X_train_df['stroke'] == 1]

minority_upsampled = resample(minority, replace=True, 
                               n_samples=len(majority), 
                               random_state=RANDOM_STATE)

train_balanced = pd.concat([majority, minority_upsampled])
X_train_bal = train_balanced.drop(columns=['stroke'])
y_train_bal = train_balanced['stroke']

print(f'Treino balanceado: {len(X_train_bal)} amostras')
print(f'Proporção AVC após balanceamento: {y_train_bal.mean()*100:.1f}%')

# Normalização (para Regressão Logística)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_bal)
X_val_scaled   = scaler.transform(X_val)
X_test_scaled  = scaler.transform(X_test)

---
## 4. Modelagem e Treinamento

### 4.1 Modelo 1 — Regressão Logística (Baseline)

In [ ]:
lr = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
lr.fit(X_train_scaled, y_train_bal)

y_pred_lr = lr.predict(X_val_scaled)
y_prob_lr = lr.predict_proba(X_val_scaled)[:, 1]

print('=== Regressão Logística — Validação ===')
print(classification_report(y_val, y_pred_lr, target_names=['Sem AVC', 'Com AVC']))
print(f'ROC-AUC: {roc_auc_score(y_val, y_prob_lr):.4f}')

### 4.2 Modelo 2 — Random Forest

In [ ]:
rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    min_samples_leaf=5,
    class_weight='balanced',
    random_state=RANDOM_STATE,
    n_jobs=-1
)
rf.fit(X_train_bal, y_train_bal)

y_pred_rf = rf.predict(X_val)
y_prob_rf = rf.predict_proba(X_val)[:, 1]

print('=== Random Forest — Validação ===')
print(classification_report(y_val, y_pred_rf, target_names=['Sem AVC', 'Com AVC']))
print(f'ROC-AUC: {roc_auc_score(y_val, y_prob_rf):.4f}')

### 4.3 Modelo 3 — Gradient Boosting

In [ ]:
gb = GradientBoostingClassifier(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=4,
    subsample=0.8,
    random_state=RANDOM_STATE
)
gb.fit(X_train_bal, y_train_bal)

y_pred_gb = gb.predict(X_val)
y_prob_gb = gb.predict_proba(X_val)[:, 1]

print('=== Gradient Boosting — Validação ===')
print(classification_report(y_val, y_pred_gb, target_names=['Sem AVC', 'Com AVC']))
print(f'ROC-AUC: {roc_auc_score(y_val, y_prob_gb):.4f}')

---
## 5. Avaliação e Comparação dos Modelos

### 5.1 Comparação de métricas na validação

In [ ]:
models_val = {
    'Regressão Logística': (y_pred_lr, y_prob_lr),
    'Random Forest':       (y_pred_rf, y_prob_rf),
    'Gradient Boosting':   (y_pred_gb, y_prob_gb),
}

results = []
for name, (y_pred, y_prob) in models_val.items():
    results.append({
        'Modelo': name,
        'Accuracy': accuracy_score(y_val, y_pred),
        'Recall (AVC)': recall_score(y_val, y_pred),
        'Precision (AVC)': precision_score(y_val, y_pred),
        'F1-Score (AVC)': f1_score(y_val, y_pred),
        'ROC-AUC': roc_auc_score(y_val, y_prob)
    })

df_results = pd.DataFrame(results).set_index('Modelo')
print('=== Comparação na Validação ===')
print(df_results.round(4).to_string())

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
metrics = ['Accuracy', 'Recall (AVC)', 'Precision (AVC)', 'F1-Score (AVC)', 'ROC-AUC']
x = np.arange(len(metrics))
width = 0.25
colors_models = ['#3498db', '#e74c3c', '#2ecc71']

for i, (model_name, row) in enumerate(df_results.iterrows()):
    values = [row[m] for m in metrics]
    bars = ax.bar(x + i*width, values, width, label=model_name, color=colors_models[i],
                  alpha=0.85, edgecolor='white')
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                f'{val:.2f}', ha='center', va='bottom', fontsize=8, fontweight='bold')

ax.set_xticks(x + width)
ax.set_xticklabels(metrics, fontsize=11)
ax.set_ylim(0, 1.15)
ax.set_title('Comparação de Métricas por Modelo (Validação)', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.set_ylabel('Score')
ax.axhline(y=1.0, color='gray', linestyle='--', alpha=0.4)

plt.tight_layout()
plt.savefig('../outputs/05_comparacao_modelos.png', bbox_inches='tight')
plt.show()

### 5.2 Curvas ROC

In [ ]:
fig, ax = plt.subplots(figsize=(8, 7))

for (name, (_, y_prob)), color in zip(models_val.items(), colors_models):
    fpr, tpr, _ = roc_curve(y_val, y_prob)
    auc = roc_auc_score(y_val, y_prob)
    ax.plot(fpr, tpr, color=color, lw=2, label=f'{name} (AUC = {auc:.3f})')

ax.plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Random (AUC = 0.500)')
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate (Recall)', fontsize=12)
ax.set_title('Curvas ROC — Comparação dos Modelos', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../outputs/06_curvas_roc.png', bbox_inches='tight')
plt.show()

### 5.3 Matrizes de Confusão

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, (name, (y_pred, _)) in zip(axes, models_val.items()):
    cm = confusion_matrix(y_val, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Sem AVC', 'Com AVC'])
    disp.plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(name, fontweight='bold', fontsize=12)
    # Destacar falsos negativos
    ax.add_patch(plt.Rectangle((-0.5, 0.5), 1, 1, fill=False, edgecolor='red', lw=3))
    ax.text(0, 1.65, 'FN: Risco clínico!', ha='center', color='red', fontsize=9, fontweight='bold')

plt.suptitle('Matrizes de Confusão (Validação) — Quadrado vermelho = Falsos Negativos', 
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('../outputs/07_matrizes_confusao.png', bbox_inches='tight')
plt.show()

---
## 6. Feature Importance — Interpretação dos Modelos

### 6.1 Random Forest — Feature Importance

In [ ]:
feature_names = X_train_bal.columns.tolist()

# Random Forest importances
fi_rf = pd.Series(rf.feature_importances_, index=feature_names).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(10, 8))
colors_fi = ['#e74c3c' if v > fi_rf.quantile(0.75) else '#3498db' for v in fi_rf.values]
fi_rf.plot(kind='barh', ax=ax, color=colors_fi, edgecolor='white')
ax.set_title('Random Forest — Feature Importance\n(Vermelho = Top 25% mais importantes)', 
             fontsize=13, fontweight='bold')
ax.set_xlabel('Importância Média (Impurity-based)')
ax.axvline(fi_rf.quantile(0.75), color='red', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.savefig('../outputs/08_feature_importance_rf.png', bbox_inches='tight')
plt.show()

print('\nTop 5 features mais importantes (Random Forest):')
print(fi_rf.sort_values(ascending=False).head(5))

### 6.2 Gradient Boosting — Feature Importance

In [ ]:
fi_gb = pd.Series(gb.feature_importances_, index=feature_names).sort_values(ascending=True)

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# GB Feature Importance
colors_fi_gb = ['#e74c3c' if v > fi_gb.quantile(0.75) else '#2ecc71' for v in fi_gb.values]
fi_gb.plot(kind='barh', ax=axes[0], color=colors_fi_gb, edgecolor='white')
axes[0].set_title('Gradient Boosting — Feature Importance', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Importância')

# Coeficientes Regressão Logística
coef_lr = pd.Series(np.abs(lr.coef_[0]), index=feature_names).sort_values(ascending=True)
colors_lr = ['#e74c3c' if v > coef_lr.quantile(0.75) else '#9b59b6' for v in coef_lr.values]
coef_lr.plot(kind='barh', ax=axes[1], color=colors_lr, edgecolor='white')
axes[1].set_title('Regressão Logística — |Coeficientes|', fontsize=12, fontweight='bold')
axes[1].set_xlabel('|Coeficiente|')

plt.suptitle('Interpretação dos Modelos — Importância das Variáveis', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../outputs/09_feature_importance_comparacao.png', bbox_inches='tight')
plt.show()

### 6.3 Análise de sensibilidade por threshold (otimização do Recall)

In [ ]:
# Análise de threshold para maximizar recall no contexto clínico
thresholds = np.arange(0.1, 0.9, 0.05)

fig, ax = plt.subplots(figsize=(12, 6))

for name, (_, y_prob), color in zip(models_val.keys(), models_val.values(), colors_models):
    recalls, precisions, f1s = [], [], []
    for t in thresholds:
        y_pred_t = (y_prob >= t).astype(int)
        recalls.append(recall_score(y_val, y_pred_t, zero_division=0))
        precisions.append(precision_score(y_val, y_pred_t, zero_division=0))
        f1s.append(f1_score(y_val, y_pred_t, zero_division=0))
    ax.plot(thresholds, recalls, color=color, lw=2, label=f'{name} — Recall', linestyle='-')
    ax.plot(thresholds, f1s, color=color, lw=1.5, linestyle='--', alpha=0.6)

ax.axvline(0.3, color='black', linestyle=':', alpha=0.7, label='Threshold 0.3 (sugerido)')
ax.set_xlabel('Threshold de Classificação', fontsize=12)
ax.set_ylabel('Score', fontsize=12)
ax.set_title('Recall (linha sólida) e F1 (tracejado) por Threshold\nEscolher threshold baixo prioriza Recall (segurança clínica)', 
             fontsize=12, fontweight='bold')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../outputs/10_threshold_analysis.png', bbox_inches='tight')
plt.show()

---
## 7. Avaliação Final no Conjunto de Teste

In [ ]:
# Modelo campeão: Random Forest (melhor equilíbrio recall/F1)
# Threshold reduzido para 0.3 para priorizar recall clínico
THRESHOLD = 0.3

y_prob_rf_test = rf.predict_proba(X_test)[:, 1]
y_pred_rf_test = (y_prob_rf_test >= THRESHOLD).astype(int)

print('=' * 55)
print(f'AVALIAÇÃO FINAL — Random Forest (threshold={THRESHOLD})')
print('=' * 55)
print(classification_report(y_test, y_pred_rf_test, target_names=['Sem AVC', 'Com AVC']))
print(f'ROC-AUC: {roc_auc_score(y_test, y_prob_rf_test):.4f}')

cm_final = confusion_matrix(y_test, y_pred_rf_test)
tn, fp, fn, tp = cm_final.ravel()
print(f'\nDetalhamento:')
print(f'  Verdadeiros Negativos (TN): {tn} — Pacientes sem AVC corretamente identificados')
print(f'  Falsos Positivos (FP):      {fp} — Alarmes falsos (custo: exames extras)')
print(f'  Falsos Negativos (FN):      {fn} — RISCO: pacientes com AVC não detectados!')
print(f'  Verdadeiros Positivos (TP): {tp} — Pacientes com AVC corretamente alertados')

In [ ]:
# Comparação final treino x teste (verificar overfitting)
y_prob_rf_train = rf.predict_proba(X_train_bal)[:, 1]
y_pred_rf_train = (y_prob_rf_train >= THRESHOLD).astype(int)

final_comparison = pd.DataFrame({
    'Conjunto': ['Treino (balanceado)', 'Teste'],
    'Accuracy': [
        accuracy_score(y_train_bal, y_pred_rf_train),
        accuracy_score(y_test, y_pred_rf_test)
    ],
    'Recall': [
        recall_score(y_train_bal, y_pred_rf_train),
        recall_score(y_test, y_pred_rf_test)
    ],
    'F1-Score': [
        f1_score(y_train_bal, y_pred_rf_train),
        f1_score(y_test, y_pred_rf_test)
    ],
    'ROC-AUC': [
        roc_auc_score(y_train_bal, y_prob_rf_train),
        roc_auc_score(y_test, y_prob_rf_test)
    ]
})
print(final_comparison.round(4).to_string(index=False))

---
## 8. Discussão Crítica dos Resultados

### Qual métrica usar?
Em diagnósticos médicos de doenças graves como AVC, o **Recall** (sensibilidade) é a métrica mais crítica. Um **Falso Negativo** — dizer que um paciente não tem risco quando ele tem — pode resultar em morte ou sequelas graves. Por isso, priorizamos Recall mesmo que isso aumente os Falsos Positivos (que geram custos com exames adicionais, mas são recuperáveis).

### O modelo pode ser usado na prática?
**Sim, mas como ferramenta de suporte — nunca como diagnóstico final.**

O modelo pode ser integrado num sistema de triagem para:
- Sinalizar pacientes de alto risco para avaliação prioritária
- Apoiar médicos de emergência com grande volume de pacientes
- Identificar padrões populacionais para prevenção

**O médico SEMPRE deve ter a palavra final.** O modelo é um apoio, não um substituto.

### Limitações
- Dataset pequeno (5.110 registros) — modelos de produção precisam de muito mais dados
- Desbalanceamento severo (4,87% positivos) — mesmo com oversampling, há limitações
- `smoking_status` com 30% de dados desconhecidos compromete essa feature
- Ausência de variáveis clínicas importantes (colesterol, pressão arterial, histórico familiar)

### Principais fatores de risco identificados
Os modelos consistentemente apontam **idade**, **glicose média** e **IMC** como as variáveis mais preditivas, alinhado com a literatura médica sobre fatores de risco para AVC.

---
## 9. Resumo Final

In [ ]:
print('=' * 60)
print('RESUMO DO PROJETO — PREVISÃO DE RISCO DE AVC')
print('=' * 60)
print(f'Dataset: {df.shape[0]} pacientes, {df.shape[1]} features')
print(f'Prevalência de AVC: {df["stroke"].mean()*100:.2f}%')
print()
print('Modelos treinados:')
print('  1. Regressão Logística (baseline)')
print('  2. Random Forest (modelo principal)')
print('  3. Gradient Boosting (modelo desafiante)')
print()
print('Estratégias de balanceamento:')
print('  - Oversampling (resample) da classe minoritária no treino')
print('  - class_weight="balanced" no Random Forest')
print('  - Threshold ajustado para 0.3 (priorizar recall)')
print()
print('Métricas finais (Random Forest, teste):')
print(f'  Accuracy:  {accuracy_score(y_test, y_pred_rf_test):.4f}')
print(f'  Recall:    {recall_score(y_test, y_pred_rf_test):.4f}')
print(f'  F1-Score:  {f1_score(y_test, y_pred_rf_test):.4f}')
print(f'  ROC-AUC:   {roc_auc_score(y_test, y_prob_rf_test):.4f}')
print()
print('Top features preditivas: age, avg_glucose_level, bmi')
print('Gráficos salvos em: ../outputs/')
print('=' * 60)

---
## 10. Simulador de Risco de AVC

**Preencha os dados do paciente na célula abaixo e execute para ver a probabilidade de AVC.**


In [ ]:
# ============================================================
# SIMULADOR DE RISCO DE AVC
# Preencha os campos abaixo e execute a célula (Shift+Enter)
# ============================================================

# ---- PREENCHA AQUI ----------------------------------------
genero          = "Female"       # "Male" ou "Female"
idade           = 67             # número inteiro (anos)
hipertensao     = 0              # 0 = Não | 1 = Sim
doenca_cardiaca = 1              # 0 = Não | 1 = Sim
casado          = "Yes"          # "Yes" ou "No"
trabalho        = "Private"      # "Private", "Self-employed", "Govt_job", "children", "Never_worked"
residencia      = "Urban"        # "Urban" ou "Rural"
glicose         = 228.69         # glicose média no sangue (mg/dL)
bmi             = 36.6           # Índice de Massa Corporal
tabagismo       = "formerly smoked"  # "never smoked", "formerly smoked", "smokes", "Unknown"
# -----------------------------------------------------------

import pandas as pd
import numpy as np

le_map = {"Male": 1, "Female": 0, "Yes": 1, "No": 0, "Urban": 1, "Rural": 0}

paciente = {
    "gender":             le_map[genero],
    "age":                idade,
    "hypertension":       hipertensao,
    "heart_disease":      doenca_cardiaca,
    "ever_married":       le_map[casado],
    "Residence_type":     le_map[residencia],
    "avg_glucose_level":  glicose,
    "bmi":                bmi,
    "work_type_Govt_job":        1 if trabalho == "Govt_job" else 0,
    "work_type_Never_worked":    1 if trabalho == "Never_worked" else 0,
    "work_type_Private":         1 if trabalho == "Private" else 0,
    "work_type_Self-employed":   1 if trabalho == "Self-employed" else 0,
    "work_type_children":        1 if trabalho == "children" else 0,
    "smoking_status_Unknown":         1 if tabagismo == "Unknown" else 0,
    "smoking_status_formerly smoked": 1 if tabagismo == "formerly smoked" else 0,
    "smoking_status_never smoked":    1 if tabagismo == "never smoked" else 0,
    "smoking_status_smokes":          1 if tabagismo == "smokes" else 0,
}

X_pac = pd.DataFrame([paciente])
for col in X_train_bal.columns:
    if col not in X_pac.columns:
        X_pac[col] = 0
X_pac = X_pac[X_train_bal.columns]

prob = rf.predict_proba(X_pac)[0][1]
pred = int(prob >= 0.3)

print("=" * 50)
print("RESULTADO DA AVALIAÇÃO DE RISCO")
print("=" * 50)
print()
print(f"  Paciente : {genero}, {idade} anos")
print(f"  Glicose  : {glicose} mg/dL")
print(f"  IMC      : {bmi}")
print(f"  Hipert.  : {'Sim' if hipertensao else 'Não'}  |  Cardíaca: {'Sim' if doenca_cardiaca else 'Não'}")
print(f"  Tabag.   : {tabagismo}")
print()
print(f"  Probabilidade de AVC : {prob*100:.1f}%")
print()
if prob >= 0.6:
    print("RISCO ALTO — Encaminhar para avaliação médica urgente")
elif prob >= 0.3:
    print("RISCO MODERADO — Recomenda-se avaliação médica")
else:
    print("RISCO BAIXO — Manter acompanhamento de rotina")
print()
print("=" * 50)
print("Este resultado é apenas um suporte à decisão.")
print("O médico deve sempre ter a palavra final.")
print("=" * 50)
